# Simulate rates and retrieve the paths

A curve describes today's discount factors. A model describes how future rates
can evolve consistently with a specified market setup. Here we configure a
one-factor Hull–White model, generate scenarios, retrieve every path as a pandas
table, and summarize the distribution. [Setup](../README.md).

These are model-implied pricing scenarios, not forecasts. The values returned
by this Hull–White simulation are **short rates in decimal units**, not discount
factors, swap PVs, or exposures. Converting market scenarios into future trade
values is another calculation, illustrated in [XVA](../cva/xva.ipynb).

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import quantsupport as qs
from market_data import load_market, read_configuration

market = load_market()
ref = market.quotes.reference_date
simulation_spec = read_configuration(
    "hullwhite",
    "simulation.json",
)
simulation_spec


## Read the model specification as a set of financial choices

[simulation.json](data/simulation.json) contains the nested serialization
schema so it does not dominate the example. Its fields mean:

| Field | Meaning in this example |
| --- | --- |
| `market_index` | SOFR identifies the curve being simulated |
| `HullWhite.alpha` | Mean reversion, 0.05 per year |
| `parameter_source.Calibrated` | Fit a volatility schedule to the constructed market |
| `source.Surface` / `strike: Atm` | Use ATM caplet prices inferred from the SOFR Black surface |
| `horizon` / `frequency` | Five years of monthly output dates |
| `n_paths` / `seed` | 2,048 paths and a reproducible random-number sequence |
| `day_counter` | Actual/365 time coordinates |

The calibration translates Black caplet quotes into an absolute short-rate
volatility schedule. Substituting a quoted Black vol directly as `sigma` would
change the meaning of the model. You can change the path count or observation
frequency independently of the calibration source.

In [ ]:
simulation_config = qs.SimulationConfiguration.from_dict(
    {
        **simulation_spec,
        "market_index": qs.MarketIndex.SOFR,
        "frequency": qs.Frequency.Monthly,
    }
)


def simulate(configuration):
    context = qs.PricingContext(
        quotes=market.quotes,
        curves=market.curves,
        csa_policy=market.csa_policy,
        volatility_surfaces=market.volatility_surfaces,
        simulations=[configuration],
    )
    context.initialize()

    with context:
        simulation = context.simulation(qs.MarketIndex.SOFR)
        print(
            simulation,
            "| average time step:",
            simulation.dt,
        )
        return simulation.paths(), simulation.dates()


paths, dates = simulate(simulation_config)
print(
    "Rows are paths; columns are dates:",
    paths.shape,
)
paths.iloc[:5, :6]


## Get one trajectory, one date, or the entire distribution

`paths()` copies the results into a DataFrame: a row is one trajectory and a
column is an ISO date. `dates()` returns the corresponding `qs.Date` values.
This simulation grid starts **after** the reference date; it contains no
time-zero column. `dt` is the average spacing in years. Calendar months have
different lengths, so use the actual dates for time integration.

Once copied out, these pandas values remain usable after the context closes.
`ctx.simulations()` lists all configured simulations; `ctx.simulation(index)`
selects one. Requesting an unconfigured index raises an error.

In [ ]:
one_path = paths.iloc[0]  # All dates on the first trajectory.
terminal_rates = paths.iloc[:, -1]  # All trajectories at the last date.
summary = pd.DataFrame(
    {
        "mean": paths.mean(axis=0),
        "std_dev": paths.std(axis=0),
        "p05": paths.quantile(0.05),
        "median": paths.quantile(0.50),
        "p95": paths.quantile(0.95),
    }
)
summary.index = pd.to_datetime(summary.index)
summary.tail()


In [ ]:
fig, axes = plt.subplots(
    1,
    2,
    figsize=(12, 4),
)
plot_dates = pd.to_datetime(paths.columns)
axes[0].plot(
    plot_dates,
    paths.iloc[:12].T * 100,
    alpha=0.35,
    linewidth=0.8,
)
axes[0].plot(
    summary.index,
    summary["mean"] * 100,
    color="black",
    label="Mean",
)
axes[0].fill_between(
    summary.index,
    summary["p05"] * 100,
    summary["p95"] * 100,
    alpha=0.15,
    label="5th–95th percentiles",
)
axes[0].set(
    title="Hull–White short-rate paths",
    xlabel="Simulation date",
    ylabel="Short rate (%)",
)
axes[0].legend()
axes[1].hist(
    terminal_rates * 100,
    bins=35,
)
axes[1].set(
    title=f"Distribution on {dates[-1]}",
    xlabel="Short rate (%)",
    ylabel="Number of paths",
)
fig.autofmt_xdate()
plt.tight_layout()
plt.show()


The shaded band is a cross-sectional range of simulated rates at each date,
not a confidence interval for the mean and not portfolio PFE. Hull–White is
Gaussian, so negative short rates are possible. A sample path need not resemble
today's forward curve; the initial curve constrains pricing, not each draw.

## Reproducibility and Monte Carlo precision

Holding inputs, model, grid, path count, seed, and library version fixed should
reproduce the paths. A different seed changes sampling error. More paths reduce
sampling error, while changing the time grid addresses discretization; neither
fixes a poor model choice. The terminal mean's standard error is estimated from
the sample standard deviation divided by the square root of the path count.

In [ ]:
repeated, _ = simulate(simulation_config)
other_config = qs.SimulationConfiguration.from_dict({**simulation_spec, "seed": 43})
other_paths, _ = simulate(other_config)
pd.DataFrame(
    [
        {
            "seed": seed,
            "matches_first_run": frame.equals(paths),
            "terminal_mean": frame.iloc[:, -1].mean(),
            "mean_standard_error": frame.iloc[:, -1].std() / len(frame) ** 0.5,
        }
        for seed, frame in [(42, paths), (42, repeated), (43, other_paths)]
    ]
)


For a larger run, construct another `SimulationConfiguration` with a larger
`n_paths`. Retrieving paths materializes the entire matrix, so its memory use
grows with both paths and dates. Export selected results with ordinary pandas,
for example `summary.to_csv("rate_summary.csv")`.

Do not treat a terminal-rate percentile as a trade-value percentile. The
[XVA notebook](../cva/xva.ipynb) builds future cashflow valuations and netting-set
exposures, while [scripting](../scripting/scripted_cashflows.ipynb) explains how dated payoff
events obtain path-averaged values from rate models.